# Supplementary Tables: Ensembl × CAT Gene Concordance

Generates two supplementary TSV tables for publication.

| Table | File | Content |
|-------|------|---------|
| S1 | supp_table_s1_gene_concordance_summary.tsv | Per-gene summary across all 462 assemblies |
| S2 | supp_table_s2_per_assembly_gene_pairs.tsv | Per-assembly × gene-pair detail |

**Input:** Pipeline output directory (set OUTPUT_DIR below)

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path(os.getenv('HPRC_QC_OUTPUT_DIR',
    '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))

QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUPP_DIR = OUTPUT_DIR / 'supplementary_tables'
SUPP_DIR.mkdir(parents=True, exist_ok=True)

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"QC_DIR exists: {QC_DIR.exists()}")
print(f"RESULTS_DIR exists: {RESULTS_DIR.exists()}")

## Load per-assembly data

In [ ]:
# Load all transcript concordance files
transcript_concordance_frames = []
for accession_dir in sorted(QC_DIR.iterdir()):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}_transcript_concordance.tsv'
    try:
        df = pd.read_csv(fp, sep='\t')
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        transcript_concordance_frames.append(df)
    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")

transcript_concordance_df = pd.concat(transcript_concordance_frames, ignore_index=True)
print(f"Loaded transcript concordance: {len(transcript_concordance_df):,} rows from {len(transcript_concordance_frames)} assemblies")

In [ ]:
# Load all coding integrity files
coding_integrity_frames = []
for accession_dir in sorted(QC_DIR.iterdir()):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}_coding_integrity.tsv'
    try:
        df = pd.read_csv(fp, sep='\t')
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        coding_integrity_frames.append(df)
    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")

coding_integrity_df = pd.concat(coding_integrity_frames, ignore_index=True)
print(f"Loaded coding integrity: {len(coding_integrity_df):,} rows from {len(coding_integrity_frames)} assemblies")

In [ ]:
import gc

# Load all gene presence files — streaming to avoid OOM with 462 assemblies.
# Each file is read, aggregated, then freed; only small summaries are kept.
GENE_PRESENCE_COLS = {'assembly_accession', 'ensembl_gene_id', 'gene_name',
                      'present_in_ensembl', 'present_in_cat', 'ensembl_biotype'}

def safe_mode(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

partial_agg_frames = []
lookup_frames = []
n_loaded = 0

for accession_dir in sorted(QC_DIR.iterdir()):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}_gene_presence.tsv'
    try:
        df = pd.read_csv(fp, sep='\t', usecols=lambda c: c in GENE_PRESENCE_COLS)
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue

        # Minimal lookup records: (assembly_accession, ensembl_gene_id) -> gene_name
        lk = df[['assembly_accession', 'ensembl_gene_id', 'gene_name']].dropna(
            subset=['ensembl_gene_id', 'gene_name'])
        lookup_frames.append(lk)

        # Per-gene presence aggregation for this file, then discard full data
        df['is_both'] = df['present_in_ensembl'].astype(bool) & df['present_in_cat'].astype(bool)
        df['is_ensembl_only'] = df['present_in_ensembl'].astype(bool) & ~df['present_in_cat'].astype(bool)
        df['is_cat_only'] = ~df['present_in_ensembl'].astype(bool) & df['present_in_cat'].astype(bool)

        pagg = df.groupby('gene_name', sort=False).agg(
            n_assessed=('is_both', 'count'),
            n_both=('is_both', 'sum'),
            n_ensembl_only=('is_ensembl_only', 'sum'),
            n_cat_only=('is_cat_only', 'sum'),
            ensembl_biotype=('ensembl_biotype', safe_mode),
        ).reset_index()
        partial_agg_frames.append(pagg)
        n_loaded += 1
        del df, pagg, lk
        gc.collect()

    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")

print(f"Processed {n_loaded} gene presence files; building aggregations...")

# ens_to_gene: (assembly_accession, ensembl_gene_id) -> gene_name lookup table
ens_to_gene = (pd.concat(lookup_frames, ignore_index=True)
               .drop_duplicates(subset=['assembly_accession', 'ensembl_gene_id']))
for col in ('assembly_accession', 'ensembl_gene_id', 'gene_name'):
    ens_to_gene[col] = ens_to_gene[col].astype('category')
del lookup_frames
gc.collect()
print(f"ens_to_gene: {len(ens_to_gene):,} rows, "
      f"{ens_to_gene.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# gp_lookup used in S2 cell to annotate RBH pairs with gene_name
gp_lookup = ens_to_gene.set_index(['assembly_accession', 'ensembl_gene_id'])['gene_name']

# Gene-level presence aggregation (gp_agg) — used in S1 cell
all_partial = pd.concat(partial_agg_frames, ignore_index=True)
del partial_agg_frames
gc.collect()

gp_agg = all_partial.groupby('gene_name').agg(
    n_assemblies_assessed=('n_assessed', 'sum'),
    n_assemblies_both=('n_both', 'sum'),
    n_assemblies_ensembl_only=('n_ensembl_only', 'sum'),
    n_assemblies_cat_only=('n_cat_only', 'sum'),
    ensembl_biotype=('ensembl_biotype', safe_mode),
).reset_index()
gp_agg['pct_assemblies_both'] = (
    gp_agg['n_assemblies_both'] / gp_agg['n_assemblies_assessed'] * 100
).round(1)

del all_partial
gc.collect()
print(f"gp_agg: {len(gp_agg):,} genes")

In [ ]:
# Load all grch38_divergence files (may not exist for all assemblies — skip missing)
divergence_frames = []
for accession_dir in sorted(QC_DIR.iterdir()):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}_grch38_divergence.tsv'
    if not fp.exists():
        continue
    try:
        df = pd.read_csv(fp, sep='\t')
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        divergence_frames.append(df)
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")

if divergence_frames:
    divergence_df = pd.concat(divergence_frames, ignore_index=True)
    print(f"Loaded grch38 divergence: {len(divergence_df):,} rows from {len(divergence_frames)} assemblies")
else:
    divergence_df = pd.DataFrame(columns=['assembly_accession', 'sample_name', 'ensembl_gene_id',
                                           'cat_gene_id', 'gene_name', 'ensembl_biotype',
                                           'ref_biotype', 'divergence_category'])
    print("WARNING: no grch38 divergence files found; divergence_df is empty")

## Build Supplementary Table S2: Per-assembly gene-pair detail

In [ ]:
# Load all RBH gene pair files
rbh_frames = []
for accession_dir in sorted(RESULTS_DIR.iterdir()):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}.gene_pairs_rbh.tsv'
    try:
        df = pd.read_csv(fp, sep='\t')
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        rbh_frames.append(df)
    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")

rbh_df = pd.concat(rbh_frames, ignore_index=True)
print(f"Loaded RBH gene pairs: {len(rbh_df):,} rows from {len(rbh_frames)} assemblies")

# Keep only RBH pairs
rbh_df = rbh_df[rbh_df['is_rbh'] == True].copy()
print(f"After filtering is_rbh=True: {len(rbh_df):,} rows")

# Join transcript concordance onto rbh_df via (assembly_accession, ensembl_gene_id, cat_gene_id)
join_cols = ['assembly_accession', 'ensembl_gene_id', 'cat_gene_id']
tc_cols = join_cols + [
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index'
]
tc_subset = transcript_concordance_df[tc_cols].drop_duplicates(subset=join_cols)
merged = rbh_df.merge(tc_subset, on=join_cols, how='left')
print(f"After joining transcript concordance: {len(merged):,} rows")

# Annotate gene_name using gp_lookup (pre-computed in gene presence cell)
merged['gene_name'] = [
    gp_lookup.get((row.assembly_accession, row.ensembl_gene_id), None)
    for row in merged[['assembly_accession', 'ensembl_gene_id']].itertuples()
]

# Join coding integrity
ci_cols = join_cols + ['classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected']
ci_subset = coding_integrity_df[ci_cols].drop_duplicates(subset=join_cols)
ci_subset = ci_subset.rename(columns={'classification': 'cds_classification'})
merged = merged.merge(ci_subset, on=join_cols, how='left')
print(f"After joining coding integrity: {len(merged):,} rows")

# Join grch38 divergence
div_cols = join_cols + ['divergence_category']
if not divergence_df.empty:
    div_subset = divergence_df[div_cols].drop_duplicates(subset=join_cols)
    merged = merged.merge(div_subset, on=join_cols, how='left')
else:
    merged['divergence_category'] = None
print(f"After joining divergence: {len(merged):,} rows")

# Compute exact pct columns (handle div by zero)
merged['ens_to_cat_exact_pct'] = np.where(
    merged['n_ensembl_transcripts'] > 0,
    (merged['n_ens_exact'] / merged['n_ensembl_transcripts'] * 100).round(1),
    np.nan
)
merged['cat_to_ens_exact_pct'] = np.where(
    merged['n_cat_transcripts'] > 0,
    (merged['n_cat_exact'] / merged['n_cat_transcripts'] * 100).round(1),
    np.nan
)

# Select and order final columns
s2_cols = [
    'assembly_accession',
    'sample_name',
    'ensembl_gene_id',
    'cat_gene_id',
    'gene_name',
    'ensembl_biotype',
    'cat_biotype',
    'frac_ensembl_covered',
    'frac_cat_covered',
    'n_ensembl_transcripts',
    'n_cat_transcripts',
    'n_ens_exact',
    'n_cat_exact',
    'ens_to_cat_exact_pct',
    'cat_to_ens_exact_pct',
    'ens_to_cat_concordance_rate',
    'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
    'cds_classification',
    'start_codon_match',
    'stop_codon_match',
    'frameshift_detected',
    'divergence_category',
]
# Only keep columns that exist
s2_cols = [c for c in s2_cols if c in merged.columns]
s2_df = merged[s2_cols].copy()

out_s2 = SUPP_DIR / 'supp_table_s2_per_assembly_gene_pairs.tsv'
s2_df.to_csv(out_s2, sep='\t', index=False)
print(f"\nSaved S2: {out_s2}")
print(f"Shape: {s2_df.shape}")
s2_df.head(3)

## Build Supplementary Table S1: Gene-level concordance summary

In [ ]:
# gp_agg, ens_to_gene, and safe_mode are pre-computed in the gene presence loading cell.

# --- Transcript concordance medians ---
tc_with_gene = transcript_concordance_df.merge(
    ens_to_gene, on=['assembly_accession', 'ensembl_gene_id'], how='left'
)

# Compute per-row exact pct for concordance
tc_with_gene['ens_exact_pct'] = np.where(
    tc_with_gene['n_ensembl_transcripts'] > 0,
    tc_with_gene['n_ens_exact'] / tc_with_gene['n_ensembl_transcripts'] * 100,
    np.nan
)
tc_with_gene['cat_exact_pct'] = np.where(
    tc_with_gene['n_cat_transcripts'] > 0,
    tc_with_gene['n_cat_exact'] / tc_with_gene['n_cat_transcripts'] * 100,
    np.nan
)

tc_medians = tc_with_gene.dropna(subset=['gene_name']).groupby('gene_name').agg(
    median_ens_to_cat_exact_pct=('ens_exact_pct', 'median'),
    median_cat_to_ens_exact_pct=('cat_exact_pct', 'median'),
    median_ens_to_cat_concordance_rate=('ens_to_cat_concordance_rate', 'median'),
    median_cat_to_ens_concordance_rate=('cat_to_ens_concordance_rate', 'median'),
).reset_index().round(1)

# --- CDS integrity ---
ci_with_gene = coding_integrity_df.merge(
    ens_to_gene, on=['assembly_accession', 'ensembl_gene_id'], how='left'
)

ci_agg = ci_with_gene.dropna(subset=['gene_name']).groupby('gene_name').agg(
    n_assemblies_cds_assessed=('assembly_accession', 'nunique'),
).reset_index()

ci_bool = ci_with_gene.dropna(subset=['gene_name']).copy()
for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected']:
    if col in ci_bool.columns:
        ci_bool[col] = ci_bool[col].map({True: True, False: False,
                                          'True': True, 'False': False,
                                          1: True, 0: False})

ci_pct = ci_bool.groupby('gene_name').agg(
    _n_start=('start_codon_match', 'count'),
    _sum_start=('start_codon_match', 'sum'),
    _n_stop=('stop_codon_match', 'count'),
    _sum_stop=('stop_codon_match', 'sum'),
    _n_fs=('frameshift_detected', 'count'),
    _sum_fs=('frameshift_detected', 'sum'),
).reset_index()
ci_pct['pct_start_codon_match'] = np.where(
    ci_pct['_n_start'] > 0,
    (ci_pct['_sum_start'] / ci_pct['_n_start'] * 100).round(1),
    np.nan
)
ci_pct['pct_stop_codon_match'] = np.where(
    ci_pct['_n_stop'] > 0,
    (ci_pct['_sum_stop'] / ci_pct['_n_stop'] * 100).round(1),
    np.nan
)
ci_pct['pct_frameshift_detected'] = np.where(
    ci_pct['_n_fs'] > 0,
    (ci_pct['_sum_fs'] / ci_pct['_n_fs'] * 100).round(1),
    np.nan
)
ci_pct = ci_pct[['gene_name', 'pct_start_codon_match', 'pct_stop_codon_match', 'pct_frameshift_detected']]
ci_agg = ci_agg.merge(ci_pct, on='gene_name', how='left')

# --- Divergence ---
if not divergence_df.empty and 'gene_name' in divergence_df.columns:
    div_agg_base = divergence_df.dropna(subset=['gene_name', 'divergence_category'])
    div_counts = div_agg_base.groupby(['gene_name', 'divergence_category']).size().unstack(fill_value=0).reset_index()
    for cat in ['both_agree_reference', 'both_agree_diverged', 'ensembl_specific', 'cat_specific']:
        if cat not in div_counts.columns:
            div_counts[cat] = 0
    div_counts['_n_div_total'] = div_counts[['both_agree_reference', 'both_agree_diverged',
                                              'ensembl_specific', 'cat_specific']].sum(axis=1)
    for cat in ['both_agree_reference', 'both_agree_diverged', 'ensembl_specific', 'cat_specific']:
        div_counts[f'pct_{cat}'] = np.where(
            div_counts['_n_div_total'] > 0,
            (div_counts[cat] / div_counts['_n_div_total'] * 100).round(1),
            np.nan
        )
    div_mode = div_agg_base.groupby('gene_name')['divergence_category'].agg(safe_mode).reset_index()
    div_mode.columns = ['gene_name', 'predominant_divergence_category']
    div_final = div_counts[['gene_name', 'pct_both_agree_reference', 'pct_both_agree_diverged',
                              'pct_ensembl_specific', 'pct_cat_specific']]
    div_final = div_final.merge(div_mode, on='gene_name', how='left')
else:
    div_final = pd.DataFrame(columns=['gene_name', 'predominant_divergence_category',
                                       'pct_both_agree_reference', 'pct_both_agree_diverged',
                                       'pct_ensembl_specific', 'pct_cat_specific'])

# --- Assemble S1 ---
# gp_agg already contains: gene_name, n_assemblies_assessed, n_assemblies_both,
# n_assemblies_ensembl_only, n_assemblies_cat_only, pct_assemblies_both, ensembl_biotype
s1_df = gp_agg.copy()
s1_df = s1_df.merge(tc_medians, on='gene_name', how='left')
s1_df = s1_df.merge(ci_agg, on='gene_name', how='left')
s1_df = s1_df.merge(div_final, on='gene_name', how='left')

s1_df['predominant_divergence_category'] = s1_df['predominant_divergence_category'].fillna('N/A')

# Final column order
s1_col_order = [
    'gene_name',
    'ensembl_biotype',
    'n_assemblies_assessed',
    'n_assemblies_both',
    'n_assemblies_ensembl_only',
    'n_assemblies_cat_only',
    'pct_assemblies_both',
    'median_ens_to_cat_exact_pct',
    'median_cat_to_ens_exact_pct',
    'median_ens_to_cat_concordance_rate',
    'median_cat_to_ens_concordance_rate',
    'n_assemblies_cds_assessed',
    'pct_start_codon_match',
    'pct_stop_codon_match',
    'pct_frameshift_detected',
    'predominant_divergence_category',
    'pct_both_agree_reference',
    'pct_both_agree_diverged',
    'pct_ensembl_specific',
    'pct_cat_specific',
]
s1_col_order = [c for c in s1_col_order if c in s1_df.columns]
s1_df = s1_df[s1_col_order].sort_values('gene_name').reset_index(drop=True)

out_s1 = SUPP_DIR / 'supp_table_s1_gene_concordance_summary.tsv'
s1_df.to_csv(out_s1, sep='\t', index=False)
print(f"Saved S1: {out_s1}")
print(f"Shape: {s1_df.shape}")
s1_df.head(3)

## Summary

In [ ]:
print(f"Table S1: {len(s1_df):,} genes \u00d7 {len(s1_df.columns)} columns")
print(f"Table S2: {len(s2_df):,} gene-pair records \u00d7 {len(s2_df.columns)} columns")
print(f"\nFiles written to: {SUPP_DIR}")
for f in sorted(SUPP_DIR.glob('*.tsv')):
    size = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size:.1f} MB")